# CellTypist vs CyteType agreement with cxg labels

Run CellTypist as the clustering weak prior, annotate clusters with CyteType, then score both CellTypist and CyteType against CELLxGENE author labels (`cell_type`) with CyteOnto.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns
from dotenv import load_dotenv

from celltypist_runner import CellTypistRunnerConfig, annotate_celltypist
from cluster_validation import ClusterValidationConfig, run_cluster_validation_on_adata
from cytetype_runner import CyteTypeRunnerConfig, require_api_key, run_cytetype
from cyteonto import CyteOntoConfig, attach_cytescores_to_obs, run_cyteonto
from shared.repo import REPO_ROOT
from study_context import ExperimentContext, experiment_context_summary

load_dotenv()

In [ ]:
SRX = "SRX17412841"
AUTHOR_COL = "cell_type"
CELLTYPIST_COL = "predicted_labels"
CYTETYPE_COL = "cytetype_annotation_leiden_merged"
ALGORITHM_COLS = {"celltypist": CELLTYPIST_COL, "cytetype": CYTETYPE_COL}

OUTPUT_ROOT = REPO_ROOT / "output" / "celltypist_vs_cxg"
DATA_DIR = OUTPUT_ROOT / "data"
FIGS_DIR = REPO_ROOT / "writeups" / "celltypist_vs_cxg" / ".figs"
CONTEXTS_JSONL = REPO_ROOT / "output" / "context" / "contexts.jsonl"
LOCAL_H5AD_ROOT = REPO_ROOT / "data" / "scbasecount" / "2026-01-12" / "h5ad" / "GeneFull" / "Homo_sapiens"

DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_context(accession: str, contexts_path: Path) -> str:
    if not contexts_path.is_file():
        return ""
    for line in contexts_path.read_text().splitlines():
        if not line.strip():
            continue
        ctx = ExperimentContext.model_validate_json(line)
        if ctx.accession == accession:
            return experiment_context_summary(ctx)
    return ""

## 1. Load raw h5ad and run CellTypist

In [ ]:
raw_path = LOCAL_H5AD_ROOT / f"{SRX}.h5ad"
adata_raw = sc.read(raw_path)

celltypist_cfg = CellTypistRunnerConfig(modelName="Nuclei_Lung_Airway.pkl")
adata_raw = annotate_celltypist(adata_raw, celltypist_cfg)
adata_raw.obs[CELLTYPIST_COL].value_counts().head()

## 2. Cluster with CellTypist weak prior

In [ ]:
cluster_cfg = ClusterValidationConfig(
    weakPriorKey=CELLTYPIST_COL,
    runLabel=f"{SRX}_celltypist_prior",
    outputDir=DATA_DIR,
    figsDir=OUTPUT_ROOT / "figs",
)
adata_clustered, cluster_result = run_cluster_validation_on_adata(
    adata_raw.copy(),
    cluster_cfg,
    SRX,
    plot=False,
)
clustered_path = DATA_DIR / f"{SRX}_celltypist_prior_clustered.h5ad"
adata_clustered.write(clustered_path)
cluster_result.selectedResolution, cluster_result.nClustersPostMerge

## 3. CyteType annotation

In [ ]:
require_api_key()
study_context = load_context(SRX, CONTEXTS_JSONL)
cytetype_cfg = CyteTypeRunnerConfig(
    srxAccession=SRX,
    outputDir=DATA_DIR,
)
cytetype_result = run_cytetype(
    cytetype_cfg,
    clustered_path,
    group_key="leiden_merged",
    study_context=study_context,
)
annotated_path = cytetype_result.outputPath
annotated_path

## 4. CyteOnto: score CellTypist and CyteType against cxg

In [ ]:
cyteonto_cfg = CyteOntoConfig(
    h5adPath=annotated_path,
    authorCol=AUTHOR_COL,
    algorithmCols=ALGORITHM_COLS,
    runsDir=OUTPUT_ROOT / "cyteonto_runs",
    payloadDir=OUTPUT_ROOT / "cyteonto_payloads",
)
cyteonto_df = run_cyteonto(cyteonto_cfg)
if cyteonto_df is None:
    raise RuntimeError("CyteOnto run interrupted; call cyteonto.check_pending_runs() and reload the CSV")
cyteonto_df.head()

## 5. Analysis: celltypist vs cytetype agreement with cxg

In [ ]:
adata = sc.read_h5ad(annotated_path)
obs = adata.obs.copy()
obs = attach_cytescores_to_obs(
    obs,
    cyteonto_df,
    author_col=AUTHOR_COL,
    algorithm_col=CELLTYPIST_COL,
    algorithm="celltypist",
    out_col="cytescore_celltypist",
)
obs = attach_cytescores_to_obs(
    obs,
    cyteonto_df,
    author_col=AUTHOR_COL,
    algorithm_col=CYTETYPE_COL,
    algorithm="cytetype",
    out_col="cytescore_cytetype",
)
obs = obs.dropna(subset=["cytescore_celltypist", "cytescore_cytetype"])
obs[[AUTHOR_COL, CELLTYPIST_COL, CYTETYPE_COL, "cytescore_celltypist", "cytescore_cytetype"]].head()

In [ ]:
summary = (
    obs.groupby(AUTHOR_COL, observed=True)
    .agg(
        n_cells=(AUTHOR_COL, "size"),
        mean_cytescore_celltypist=("cytescore_celltypist", "mean"),
        mean_cytescore_cytetype=("cytescore_cytetype", "mean"),
    )
    .assign(
        delta_celltypist_minus_cytetype=lambda df: (
            df["mean_cytescore_celltypist"] - df["mean_cytescore_cytetype"]
        )
    )
    .sort_values("n_cells", ascending=False)
)
summary

In [ ]:
long = obs.melt(
    id_vars=[AUTHOR_COL],
    value_vars=["cytescore_celltypist", "cytescore_cytetype"],
    var_name="method",
    value_name="cytescore_similarity",
)
long["method"] = long["method"].map(
    {
        "cytescore_celltypist": "celltypist",
        "cytescore_cytetype": "cytetype",
    }
)
order = summary.index.tolist()

fig, ax = plt.subplots(figsize=(8, max(4, len(order) * 0.25)))
sns.boxplot(
    data=long,
    x="cytescore_similarity",
    y=AUTHOR_COL,
    hue="method",
    order=order,
    fliersize=0,
    ax=ax,
)
ax.set_xlabel("CyteScore vs cxg author label")
ax.set_ylabel("cxg cell type")
ax.set_title(f"{SRX}: cytescore by method and cxg cell type")
fig.tight_layout()
fig.savefig(FIGS_DIR / f"cytescore_by_method_{SRX}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
delta = summary["delta_celltypist_minus_cytetype"].sort_values()

fig, ax = plt.subplots(figsize=(7, max(4, len(delta) * 0.25)))
colors = ["#228B22" if v > 0 else "#d73027" for v in delta]
ax.barh(delta.index, delta.values, color=colors)
ax.axvline(0, color="grey", linewidth=0.5)
ax.set_xlabel("mean cytescore(celltypist) - mean cytescore(cytetype)")
ax.set_ylabel("cxg cell type")
ax.set_title(f"{SRX}: which method agrees more with cxg per cell type")
fig.tight_layout()
fig.savefig(FIGS_DIR / f"cytescore_delta_{SRX}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from cytetype_runner.utils import confidence_by_cluster

confidence_map = confidence_by_cluster(adata)
obs["cytetype_confidence"] = obs["leiden_merged"].astype(str).map(confidence_map)

conf_order = ["Low", "Moderate", "High"]
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(
    data=obs,
    x="cytetype_confidence",
    y="cytescore_cytetype",
    order=[c for c in conf_order if c in obs["cytetype_confidence"].unique()],
    ax=ax,
)
ax.set_xlabel("CyteType cluster confidence")
ax.set_ylabel("CyteScore vs cxg (CyteType)")
ax.set_title(f"{SRX}: CyteType confidence vs cxg agreement")
fig.tight_layout()
fig.savefig(FIGS_DIR / f"cytescore_vs_confidence_{SRX}.png", dpi=150, bbox_inches="tight")
plt.show()